# 工具的应用案例

## 1、使用args_schema

举例

In [1]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os



# 从.env文件中加载环境变量
load_dotenv(override=True)

CLOSEAI_API_KEY = os.getenv("CLOSEAI_API_KEY")
CLOSEAI_BASE_URL = os.getenv("CLOSEAI_BASE_URL")

model = init_chat_model(
    model="gpt-5.4-mini",
    model_provider="openai",
    api_key=CLOSEAI_API_KEY,
    base_url=CLOSEAI_BASE_URL
)

In [3]:

from langchain_core.utils.function_calling import convert_to_openai_tool
from pydantic import BaseModel, Field
from langchain_core.tools import tool


class WeacherSchema(BaseModel):
    city: str = Field(default="北京", description="具体的城市名称")
    if_forecast: bool = Field(default=False, description="是否包含明日天气")


#定义工具
@tool("get_weacher_and_forecast", description="查询当日的天气，可以包含明天的天气预报",
      args_schema=WeacherSchema)
def get_weather(city: str, if_forecast: bool):
    res = f"{city}今天天气不错"
    if if_forecast:
        res += "\n明天下雨"
    return res


# print(convert_to_openai_tool(get_weather))

{'type': 'function', 'function': {'name': 'get_weacher_and_forecast', 'description': '查询当日的天气，可以包含明天的天气预报', 'parameters': {'properties': {'city': {'default': '北京', 'description': '具体的城市名称', 'type': 'string'}, 'if_forecast': {'default': False, 'description': '是否包含明日天气', 'type': 'boolean'}}, 'type': 'object'}}}


In [4]:
from langchain_core.messages import HumanMessage

# 1、将工具绑定到模型上
model_with_tools = model.bind_tools([get_weather])

# 2、维护一个消息列表
messages = [HumanMessage("今天杭州的天气怎么样？明天呢？")]

# 3、调用模型,得到响应：AIMessage
response = model_with_tools.invoke(messages)

messages.append(response)

# 4、获取响应中的tool_calls字段信息
tool_calls = response.tool_calls

for tool_call in tool_calls:
    if tool_call["name"] == "get_weacher_and_forecast":
        # 5、调用工具（因为大模型不能直接调用工具，所以此时我们主动让工具调用执行）
        # 调用完，返回ToolMessage的实例
        tool_message = get_weather.invoke(tool_call)
        messages.append(tool_message)

# 6、调用模型,得到AIMessage
final_response = model.invoke(messages)

# 7、添加到消息列表中
messages.append(final_response)

# 8、遍历消息列表
for msg in messages:
    msg.pretty_print()


================================ Human Message =================================

今天杭州的天气怎么样？明天呢？
================================== Ai Message ==================================
Tool Calls:
  get_weacher_and_forecast (call_PJy3uPI6G2bkmw63bySYYigU)
 Call ID: call_PJy3uPI6G2bkmw63bySYYigU
  Args:
    city: 杭州
    if_forecast: True
================================= Tool Message =================================
Name: get_weacher_and_forecast

杭州今天天气不错
明天下雨
================================== Ai Message ==================================

杭州今天“天气不错”，明天“下雨”。

如果你愿意，我也可以顺手帮你整理成更实用的出行建议，比如要不要带伞、适合穿什么。


## 2、撰写docstring

In [5]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os

# 从.env文件中加载环境变量
load_dotenv(override=True)

CLOSEAI_API_KEY = os.getenv("CLOSEAI_API_KEY")
CLOSEAI_BASE_URL = os.getenv("CLOSEAI_BASE_URL")

model = init_chat_model(
    model="gpt-5.4-mini",
    model_provider="openai",
    api_key=CLOSEAI_API_KEY,
    base_url=CLOSEAI_BASE_URL
)

In [8]:

from langchain_core.utils.function_calling import convert_to_openai_tool
from pydantic import BaseModel, Field
from langchain_core.tools import tool

#定义工具
@tool("get_weacher_and_forecast",parse_docstring=True)
def get_weather(city: str = "北京", if_forecast: bool = False):
    """
    查询当日的天气，可以包含明天的天气预报

    Args:
        city : 城市名称
        if_forecast : 是否包含明天的天气
    """
    res = f"{city}今天天气不错"
    if if_forecast:
        res += "\n明天下雨"
    return res


print(convert_to_openai_tool(get_weather))

{'type': 'function', 'function': {'name': 'get_weacher_and_forecast', 'description': '查询当日的天气，可以包含明天的天气预报', 'parameters': {'properties': {'city': {'default': '北京', 'description': '城市名称', 'type': 'string'}, 'if_forecast': {'default': False, 'description': '是否包含明天的天气', 'type': 'boolean'}}, 'type': 'object'}}}


In [9]:
from langchain_core.messages import HumanMessage

# 1、将工具绑定到模型上
model_with_tools = model.bind_tools([get_weather])

# 2、维护一个消息列表
messages = [HumanMessage("今天杭州的天气怎么样？明天呢？")]

# 3、调用模型,得到响应：AIMessage
response = model_with_tools.invoke(messages)

messages.append(response)

# 4、获取响应中的tool_calls字段信息
tool_calls = response.tool_calls

for tool_call in tool_calls:
    if tool_call["name"] == "get_weacher_and_forecast":
        # 5、调用工具（因为大模型不能直接调用工具，所以此时我们主动让工具调用执行）
        # 调用完，返回ToolMessage的实例
        tool_message = get_weather.invoke(tool_call)
        messages.append(tool_message)

# 6、调用模型,得到AIMessage
final_response = model.invoke(messages)

# 7、添加到消息列表中
messages.append(final_response)

# 8、遍历消息列表
for msg in messages:
    msg.pretty_print()


================================ Human Message =================================

今天杭州的天气怎么样？明天呢？
================================== Ai Message ==================================
Tool Calls:
  get_weacher_and_forecast (call_2A6pcZmOz6XTuhcmZ6iOeUjq)
 Call ID: call_2A6pcZmOz6XTuhcmZ6iOeUjq
  Args:
    city: 杭州
    if_forecast: True
================================= Tool Message =================================
Name: get_weacher_and_forecast

杭州今天天气不错
明天下雨
================================== Ai Message ==================================

杭州今天天气不错，明天下雨。


## 3、多工具调用

In [15]:

from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage
from rich import print as rprint

# 1.定义工具
# 定义股票查询工具
@tool(parse_docstring=True)
def get_stock_price(company: str, timeframe: str = "today") -> str:
    """获取指定公司的股票价格信息

    Args:
        company: 公司名称（如：苹果公司, 微软公司, 谷歌公司）
        timeframe: 时间范围（today-今日, week-本周, month-本月）
    """
    # 模拟股票数据
    mock_data = {
        "苹果公司": {"today": 185.20, "week": 183.50, "month": 180.75},
        "微软公司": {"today": 415.86, "week": 412.30, "month": 405.42},
        "谷歌公司": {"today": 15.42, "week": 15.20, "month": 14.85}
    }

    if company in mock_data:
        price = mock_data[company].get(timeframe, "未知时间范围")
        return f"{company} {timeframe}价格: {price}美元"
    else:
        return f"未找到股票代码 {company} 的数据"


# 定义新闻搜索工具
@tool(parse_docstring=True)
def search_news(company: str) -> str:
    """搜索指定公司的财经新闻

    Args:
        company: 公司名称

    Returns:
        公司的财经新闻，每个新闻占一行
    """
    # 模拟新闻数据
    mock_news = {
        "苹果公司": [
            "苹果发布新款iPhone，股价上涨3%",
            "苹果与欧盟达成反垄断和解协议",
            "苹果将在印度扩大生产规模"
        ],
        "微软公司": [
            "微软Azure云业务季度增长超预期",
            "微软完成对Nuance的收购",
            "微软推出新一代AI助手Copilot"
        ],
        "谷歌公司": [
            "谷歌发布新AI模型，性能提升20%",
            "谷歌与OpenAI合作，开发新的AI助手",
            "谷歌在欧洲展开AI研究项目"
        ]
    }

    news_list = mock_news.get(company, [f"未找到{company}的相关新闻"])
    return "\n".join(news_list)


# rprint(convert_to_openai_tool(search_news))

# 2.初始化模型并绑定工具
tools = [get_stock_price, search_news]
model_with_tools = model.bind_tools(tools)

message_list = []
human_message = HumanMessage(content="苹果公司今天的股价是多少？最近有什么新闻？")
# human_message = HumanMessage(content="比较一下微软和苹果的股价")
# human_message = HumanMessage(content="腾讯最近有什么重大新闻？")
# human_message = HumanMessage(content="海水为什么是咸的？")
message_list.append(human_message)

# 3.工具调用
while True:
    response = model_with_tools.invoke(message_list)

    # rprint(response)
    # 将返回的AIMessage添加到消息列表
    message_list.append(response)

    # 如果模型不需要调用工具，直接退出循环
    if not response.tool_calls:
        print("没有工具调用，直接返回答案")
        break

    # 如果有调用工具，处理工具调用响应
    # 4.开发者根据模型的响应，调用工具并获取结果
    for tool_call in response.tool_calls:
        if tool_call["name"] == "get_stock_price":
            stock_result = get_stock_price.invoke(tool_call)
            print("stock_result", stock_result)
            message_list.append(stock_result)
        if tool_call["name"] == "search_news":
            news_result = search_news.invoke(tool_call)
            print("news_result", news_result)
            message_list.append(news_result)


# print("response", response)
# print(response.content)

for msg in message_list:
    msg.pretty_print()

stock_result content='苹果公司 today价格: 185.2美元' name='get_stock_price' tool_call_id='call_lGABSlJ7CaJLXbZonhOml2bZ'
news_result content='苹果发布新款iPhone，股价上涨3%\n苹果与欧盟达成反垄断和解协议\n苹果将在印度扩大生产规模' name='search_news' tool_call_id='call_zEhkyxCjprx8dSHsyXtbqOy4'
没有工具调用，直接返回答案
================================ Human Message =================================

苹果公司今天的股价是多少？最近有什么新闻？
================================== Ai Message ==================================
Tool Calls:
  get_stock_price (call_lGABSlJ7CaJLXbZonhOml2bZ)
 Call ID: call_lGABSlJ7CaJLXbZonhOml2bZ
  Args:
    company: 苹果公司
    timeframe: today
  search_news (call_zEhkyxCjprx8dSHsyXtbqOy4)
 Call ID: call_zEhkyxCjprx8dSHsyXtbqOy4
  Args:
    company: 苹果公司
================================= Tool Message =================================
Name: get_stock_price

苹果公司 today价格: 185.2美元
================================= Tool Message =================================
Name: search_news

苹果发布新款iPhone，股价上涨3%
苹果与欧盟达成反垄断和解协议
苹果将在印度扩大生产规模
=================

## 4、多工具调用

In [16]:
from langchain.tools import tool
from langchain.messages import HumanMessage


@tool(parse_docstring=True)
def get_weather(city: str) -> str:
    """
    获取当日天气

    Args:
        city: 城市名称
    """
    return f'{city}当天晴朗'

@tool(parse_docstring=True)
def get_news() -> str:
    """
    获取当日新闻
    """
    return "近期，受全球储蓄芯片短缺等多重因素影响，多地回收商称废旧手机回收市场迎来“火热潮”，回收价格普遍上涨，旧手机成“香饽饽”。"

model_with_tools = model.bind_tools([get_weather, get_news])

messages = [
    HumanMessage("今天杭州天气如何？今天新闻是什么？别瞎编")
]

response = model_with_tools.invoke(messages)
response.pretty_print()

================================== Ai Message ==================================
Tool Calls:
  get_weather (call_FcaXIq3sLQE2UVSoTuL8XhtB)
 Call ID: call_FcaXIq3sLQE2UVSoTuL8XhtB
  Args:
    city: 杭州
  get_news (call_ZjuPVNgbFQhJc0yXstuHWMVD)
 Call ID: call_ZjuPVNgbFQhJc0yXstuHWMVD
  Args:


In [17]:

messages.append(response)

for tool_call in response.tool_calls:
    if tool_call["name"] == "get_weather":
        tool_msg = get_weather.invoke(tool_call)
        print(tool_msg)
        messages.append(tool_msg)
    elif tool_call["name"] == "get_news":
        tool_msg = get_news.invoke(tool_call)
        print(tool_msg)
        messages.append(tool_msg)
    else:
        raise Exception("不存在的工具")

final_response = model.invoke(messages)
messages.append(final_response)

for msg in messages:
    msg.pretty_print()

content='杭州当天晴朗' name='get_weather' tool_call_id='call_FcaXIq3sLQE2UVSoTuL8XhtB'
content='近期，受全球储蓄芯片短缺等多重因素影响，多地回收商称废旧手机回收市场迎来“火热潮”，回收价格普遍上涨，旧手机成“香饽饽”。' name='get_news' tool_call_id='call_ZjuPVNgbFQhJc0yXstuHWMVD'
================================ Human Message =================================

今天杭州天气如何？今天新闻是什么？别瞎编
================================== Ai Message ==================================
Tool Calls:
  get_weather (call_FcaXIq3sLQE2UVSoTuL8XhtB)
 Call ID: call_FcaXIq3sLQE2UVSoTuL8XhtB
  Args:
    city: 杭州
  get_news (call_ZjuPVNgbFQhJc0yXstuHWMVD)
 Call ID: call_ZjuPVNgbFQhJc0yXstuHWMVD
  Args:
================================= Tool Message =================================
Name: get_weather

杭州当天晴朗
================================= Tool Message =================================
Name: get_news

近期，受全球储蓄芯片短缺等多重因素影响，多地回收商称废旧手机回收市场迎来“火热潮”，回收价格普遍上涨，旧手机成“香饽饽”。
================================== Ai Message ==================================

杭州今天天气：**晴朗**。

今天新闻：**近期受全球储蓄芯片短缺等多重因素影响，多地